# 02 - Visualize the hybrid RGB + LiDAR input

This notebook inspects the tensor consumed by the camera-depth model: 3 RGB channels, 6 sparse LiDAR channels, then splatted depth and confidence. It performs no training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Launch Jupyter from the repository root or notebooks/.")
sys.path.insert(0, str(ROOT / "src"))

from rgb_lidar_fusion.dataset import KittiObjectDepthDataset
from rgb_lidar_fusion.lidar_splatting import SplattingConfig
from rgb_lidar_fusion.model_batch import dataset_item_to_model_batch

## Build one model input

The dataset resizes RGB and calibration together to `IMAGE_SHAPE`. Splatting is local and image-space only: each retained sparse depth is copied within `SPLAT_RADIUS_PX`, with Gaussian confidence decay.

In [ ]:
DATA_ROOT = ROOT / "data/kitti/training"
FRAME_INDEX = 0
IMAGE_SHAPE = (320, 1024)
MAX_DEPTH_M = 80.0
SPLAT_RADIUS_PX = 2
SPLAT_SIGMA_PX = 1.0

dataset = KittiObjectDepthDataset(
    DATA_ROOT,
    image_shape=IMAGE_SHAPE,
    max_depth_m=MAX_DEPTH_M,
)
if not 0 <= FRAME_INDEX < len(dataset):
    raise IndexError(f"FRAME_INDEX={FRAME_INDEX} outside 0..{len(dataset) - 1}")

item = dataset[FRAME_INDEX]
splatting_config = SplattingConfig(
    radius_px=SPLAT_RADIUS_PX,
    sigma_px=SPLAT_SIGMA_PX,
)
batch = dataset_item_to_model_batch(
    item,
    include_splatted_depth=True,
    splatting_config=splatting_config,
)

rgb = item["image"]
sparse_maps = item["lidar_maps"]
hybrid_input = batch["inputs"]
enriched_lidar_maps = hybrid_input[0, 3:]
channel_names = batch["input_channels"]

print(f"sample: {item['meta']['sample_id']}")
print(f"RGB: {rgb.shape}")
print(f"sparse LiDAR maps: {sparse_maps.shape}")
print(f"enriched LiDAR maps: {enriched_lidar_maps.shape}")
print(f"model input [B, C, H, W]: {hybrid_input.shape}")

## The six sparse channels

Channels 0 to 3 are normalized by `MAX_DEPTH_M`; the plots below convert them back to meters. Only pixels selected by `point_mask` contain valid sparse measurements.

In [ ]:
sparse_names = item["meta"]["lidar_map_channels"]
point_mask = sparse_maps[5] > 0.0
point_count = int(point_mask.sum())
pixel_count = point_mask.size
print(f"valid sparse pixels: {point_count:,}/{pixel_count:,} ({100 * point_count / pixel_count:.3f}%)")
print("\nchannel                              min        mean         max")
print("-" * 63)
for channel_index, channel_name in enumerate(sparse_names):
    values = sparse_maps[channel_index][point_mask]
    if channel_index <= 3:
        values = values * MAX_DEPTH_M
    print(f"{channel_index}: {channel_name:<31} {values.min():>8.3f} {values.mean():>11.3f} {values.max():>11.3f}")

In [ ]:
valid_v, valid_u = np.nonzero(point_mask)
fig, axes = plt.subplots(2, 3, figsize=(16, 6), constrained_layout=True)
for channel_index, (axis, channel_name) in enumerate(zip(axes.flat, sparse_names)):
    axis.set_facecolor("black")
    values = sparse_maps[channel_index][point_mask]
    unit = ""
    if channel_index <= 3:
        values = values * MAX_DEPTH_M
        unit = " (m)"
    if channel_index == 5:
        axis.scatter(valid_u, valid_v, s=1.0, c="white", linewidths=0)
    else:
        points = axis.scatter(valid_u, valid_v, c=values, s=1.0, cmap="turbo", linewidths=0)
        fig.colorbar(points, ax=axis, fraction=0.035, pad=0.01)
    axis.set_title(channel_name + unit, fontsize=10)
    axis.set_xlim(0, IMAGE_SHAPE[1])
    axis.set_ylim(IMAGE_SHAPE[0], 0)
    axis.axis("off")
plt.show()

## What splatting adds

`depth_expanded` carries the normalized depth selected from nearby sparse points; `confidence` records the Gaussian distance to the selected source. These are derived context channels, not new LiDAR measurements and not a dense ground-truth depth map.

In [ ]:
depth_expanded_m = enriched_lidar_maps[6] * MAX_DEPTH_M
confidence = enriched_lidar_maps[7]
expanded_mask = confidence > 0.0
expanded_count = int(expanded_mask.sum())
print(f"sparse support:   {point_count:,} pixels ({100 * point_count / pixel_count:.3f}%)")
print(f"splatted support: {expanded_count:,} pixels ({100 * expanded_count / pixel_count:.3f}%)")
print(f"support expansion factor: {expanded_count / point_count:.2f}x")
print(f"splat configuration: radius={SPLAT_RADIUS_PX}px, sigma={SPLAT_SIGMA_PX}px")

In [ ]:
rgb_hwc = np.moveaxis(rgb, 0, 2)
sparse_depth_m = sparse_maps[0] * MAX_DEPTH_M
depth_vmax = float(np.percentile(sparse_depth_m[point_mask], 99))
fig, axes = plt.subplots(2, 2, figsize=(16, 7), constrained_layout=True)

axes[0, 0].imshow(rgb_hwc)
axes[0, 0].set_title("RGB")

axes[0, 1].set_facecolor("black")
sparse_points = axes[0, 1].scatter(
    valid_u, valid_v, c=sparse_depth_m[point_mask], s=1.0, cmap="turbo",
    vmin=0.0, vmax=depth_vmax, linewidths=0,
)
axes[0, 1].set_title("Sparse camera depth (m)")
axes[0, 1].set_xlim(0, IMAGE_SHAPE[1])
axes[0, 1].set_ylim(IMAGE_SHAPE[0], 0)
fig.colorbar(sparse_points, ax=axes[0, 1], fraction=0.035, pad=0.01)

depth_cmap = plt.colormaps["turbo"].copy()
depth_cmap.set_bad("black")
expanded_image = np.ma.masked_where(~expanded_mask, depth_expanded_m)
expanded_plot = axes[1, 0].imshow(expanded_image, cmap=depth_cmap, vmin=0.0, vmax=depth_vmax)
axes[1, 0].set_title("Splatted depth (m)")
fig.colorbar(expanded_plot, ax=axes[1, 0], fraction=0.035, pad=0.01)

confidence_plot = axes[1, 1].imshow(confidence, cmap="magma", vmin=0.0, vmax=1.0)
axes[1, 1].set_title("Splat confidence")
fig.colorbar(confidence_plot, ax=axes[1, 1], fraction=0.035, pad=0.01)
for axis in axes.flat:
    axis.axis("off")
plt.show()

## Verify the channel contract

The enriched camera-depth input has shape `[B, 11, H, W]`. Keeping the names beside the indices avoids silently training with a wrong channel order.

In [ ]:
assert hybrid_input.shape == (1, 11, *IMAGE_SHAPE)
assert len(channel_names) == hybrid_input.shape[1]
for channel_index, channel_name in enumerate(channel_names):
    print(f"{channel_index:>2}: {channel_name}")

## Next step

After projection alignment and channel semantics are understood, `03_camera_depth_training_analysis.ipynb` compares sparse-only and enriched training. During held-out validation, splatted channels are recomputed after removing target points to prevent target leakage.